# 14 — Assembly101 `v1` Download, Validation, and Smoke Video Crops

Notebook 13 produced the validated coarse TAS representation:

```text
680 assembly/disassembly sequences
350 unique raw recordings
selected view: v1
selected camera file: C10095_rgb.mp4
```

This notebook performs the next storage-sensitive stage:

```text
read notebook-13 manifests
→ fetch exact remote sizes for the required 350 v1 files
→ calculate total and remaining download size
→ select a small smoke batch by default
→ download files one by one with persistent progress
→ validate each video with FFprobe
→ verify all annotation crop intervals against video duration
→ create short low-resolution smoke crops
```

## Safe default

The default mode is:

```python
DOWNLOAD_MODE = "smoke"
SMOKE_RECORDINGS = 2
```

It downloads only two relatively small required `v1` recordings.

The notebook never downloads the other 11 camera views.

## Full-download mode

After the smoke run succeeds, change:

```python
DOWNLOAD_MODE = "full"
CONFIRM_FULL_DOWNLOAD = True
FULL_BATCH_SIZE = 10
```

Each full-mode run downloads the first incomplete batch. Rerunning the notebook continues with the next incomplete files.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 2. Install current Hugging Face download support

In [2]:
%pip install -q -U "huggingface_hub[hf_xet]" pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.7/676.7 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 56.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.


## 3. Imports

In [3]:
from pathlib import Path
from fractions import Fraction
from datetime import datetime, timezone

import json
import os
import re
import shutil
import subprocess
import time
import traceback

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from huggingface_hub import (
    HfApi,
    hf_hub_download,
    login,
    notebook_login,
    get_token,
)
from huggingface_hub.errors import (
    GatedRepoError,
    HfHubHTTPError,
    RepositoryNotFoundError,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

print("Imports OK")

Imports OK


## 4. Configuration

In [4]:
REPO_ID = "cvml-nus/assembly101"
REPO_TYPE = "dataset"
REVISION = "main"

DRIVE_ROOT = Path("/content/drive/MyDrive/mmf_tas_lab_data")
ASSEMBLY_ROOT = DRIVE_ROOT / "assembly101"

MSTCN_ROOT = (
    DRIVE_ROOT
    / "text_assisted_tas"
    / "assembly101"
    / "coarse_mstcn_format"
)

DATASET_SUMMARY_PATH = MSTCN_ROOT / "dataset_summary.json"
RECORDING_MANIFEST_PATH = (
    MSTCN_ROOT / "recording_download_manifest_v1.csv"
)
SEQUENCE_MANIFEST_PATH = (
    MSTCN_ROOT / "procedurevrl_extraction_manifest_v1.csv"
)

DOWNLOAD_RUN_ROOT = MSTCN_ROOT / "v1_download"
SMOKE_CROP_ROOT = DOWNLOAD_RUN_ROOT / "smoke_crops"

REMOTE_SIZE_MANIFEST_PATH = (
    DOWNLOAD_RUN_ROOT / "remote_size_inventory_v1.csv"
)
DOWNLOAD_STATUS_PATH = (
    DOWNLOAD_RUN_ROOT / "download_status_v1.csv"
)
SELECTED_BATCH_PATH = (
    DOWNLOAD_RUN_ROOT / "selected_download_batch_v1.csv"
)
FFPROBE_PATH = (
    DOWNLOAD_RUN_ROOT / "ffprobe_v1.csv"
)
ALIGNMENT_PATH = (
    DOWNLOAD_RUN_ROOT / "crop_alignment_validation_v1.csv"
)
SMOKE_CROP_MANIFEST_PATH = (
    DOWNLOAD_RUN_ROOT / "smoke_crop_manifest_v1.csv"
)
SUMMARY_PATH = (
    DOWNLOAD_RUN_ROOT / "assembly101_v1_download_summary.json"
)

# Valid values: "metadata_only", "smoke", "full".
DOWNLOAD_MODE = "smoke"

# Smoke mode selects relatively small recordings with both assembly and
# disassembly crops when possible.
SMOKE_RECORDINGS = 2

# Full mode downloads the first N currently incomplete files per run.
# Set to None only after reviewing total size and available storage.
FULL_BATCH_SIZE = 10
CONFIRM_FULL_DOWNLOAD = False

# One file at a time is safer for Google Drive and interrupted Colab sessions.
MAX_DOWNLOAD_ATTEMPTS = 3
RETRY_SLEEP_SECONDS = 10

# Create small previews only for smoke mode.
CREATE_SMOKE_CROPS = True
SMOKE_CROP_MAX_SECONDS = 12.0
SMOKE_CROP_HEIGHT = 360
SMOKE_CROP_CRF = 28

# Annotation crop must not exceed video duration by more than this tolerance.
DURATION_TOLERANCE_SECONDS = 1.0

SELECTED_VIEW = "v1"
SELECTED_CAMERA_FILE = "C10095_rgb.mp4"

for path in [
    ASSEMBLY_ROOT,
    DOWNLOAD_RUN_ROOT,
    SMOKE_CROP_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

if DOWNLOAD_MODE not in {"metadata_only", "smoke", "full"}:
    raise ValueError(
        "DOWNLOAD_MODE must be metadata_only, smoke, or full."
    )

print("DOWNLOAD_MODE:", DOWNLOAD_MODE)
print("ASSEMBLY_ROOT:", ASSEMBLY_ROOT)
print("MSTCN_ROOT:", MSTCN_ROOT)
print("DOWNLOAD_RUN_ROOT:", DOWNLOAD_RUN_ROOT)

DOWNLOAD_MODE: smoke
ASSEMBLY_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/assembly101
MSTCN_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format
DOWNLOAD_RUN_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download


## 5. Authenticate with Hugging Face

The notebook first checks the Colab secret `HF_TOKEN`. The token must belong to the account that was approved for Assembly101.

In [5]:
def authenticate_huggingface():
    token = None

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token:
        login(
            token=token,
            add_to_git_credential=False,
        )
        print("Authenticated using Colab secret HF_TOKEN.")
        return

    if get_token() is not None:
        print("A Hugging Face token is already available.")
        return

    print("No HF_TOKEN secret found. Opening notebook login.")
    notebook_login(skip_if_logged_in=True)


authenticate_huggingface()

if get_token() is None:
    raise RuntimeError(
        "Hugging Face authentication did not complete."
    )

A Hugging Face token is already available.


## 6. Verify Assembly101 file access and pin the repository revision

In [6]:
api = HfApi()

try:
    account = api.whoami(token=True)

    repo_info = api.dataset_info(
        repo_id=REPO_ID,
        revision=REVISION,
        files_metadata=False,
        token=True,
    )

    # Metadata visibility alone is not enough for a gated repository.
    access_test_path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        revision=repo_info.sha,
        filename="annotations/README.md",
        local_dir=ASSEMBLY_ROOT,
        token=True,
    )

    PINNED_REVISION = repo_info.sha

    print("Hugging Face user:", account.get("name"))
    print("Assembly101 gated-file access: OK")
    print("Pinned repository revision:", PINNED_REVISION)
    print("Access-test file:", access_test_path)

except GatedRepoError as exc:
    raise RuntimeError(
        "The current Hugging Face account does not have "
        "Assembly101 file access."
    ) from exc

except (RepositoryNotFoundError, HfHubHTTPError) as exc:
    raise RuntimeError(
        f"Assembly101 access check failed: {exc}"
    ) from exc

Hugging Face user: Bonart
Assembly101 gated-file access: OK
Pinned repository revision: bfc15ea5e3f0bc8f8c232af6c1b45aa137a9d967
Access-test file: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/annotations/README.md


## 7. Load notebook-13 manifests and validate the completed conversion

In [7]:
required_paths = {
    "dataset summary": DATASET_SUMMARY_PATH,
    "recording manifest": RECORDING_MANIFEST_PATH,
    "sequence/crop manifest": SEQUENCE_MANIFEST_PATH,
}

for name, path in required_paths.items():
    print(f"{name}: {path} -> exists={path.exists()}")
    if not path.exists():
        raise FileNotFoundError(
            f"Missing notebook-13 artifact: {name}: {path}"
        )

dataset_summary = json.loads(
    DATASET_SUMMARY_PATH.read_text(encoding="utf-8")
)
recordings = pd.read_csv(RECORDING_MANIFEST_PATH)
sequences = pd.read_csv(SEQUENCE_MANIFEST_PATH)

required_recording_columns = {
    "recording_name",
    "selected_view",
    "video_filename",
    "video_remote_path",
    "video_local_path",
    "video_downloaded",
    "num_sequence_crops",
}

required_sequence_columns = {
    "sequence_id",
    "split",
    "activity",
    "recording_name",
    "selected_view",
    "video_filename",
    "video_remote_path",
    "video_local_path",
    "clip_start_frame_30fps",
    "clip_end_frame_30fps_exclusive",
    "clip_start_seconds",
    "clip_end_seconds",
    "clip_duration_seconds",
    "ground_truth_path",
    "ground_truth_length_30fps",
}

missing_recording_columns = (
    required_recording_columns - set(recordings.columns)
)
missing_sequence_columns = (
    required_sequence_columns - set(sequences.columns)
)

if missing_recording_columns:
    raise KeyError(
        "Recording manifest is missing columns: "
        f"{sorted(missing_recording_columns)}"
    )

if missing_sequence_columns:
    raise KeyError(
        "Sequence manifest is missing columns: "
        f"{sorted(missing_sequence_columns)}"
    )

if dataset_summary.get("status") != "completed":
    raise RuntimeError(
        "Notebook-13 dataset summary is not completed."
    )

validation = dataset_summary.get("validation", {})
nonzero_validation = {
    key: value
    for key, value in validation.items()
    if value != 0
}

if nonzero_validation:
    raise RuntimeError(
        "Notebook 13 reported failed validation checks: "
        f"{nonzero_validation}"
    )

if recordings["recording_name"].duplicated().any():
    raise ValueError(
        "recording_download_manifest_v1.csv contains "
        "duplicate recording names."
    )

if not (
    recordings["selected_view"].astype(str) == SELECTED_VIEW
).all():
    raise ValueError("Recording manifest contains another view.")

if not (
    recordings["video_filename"].astype(str)
    == SELECTED_CAMERA_FILE
).all():
    raise ValueError("Recording manifest contains another camera file.")

print("Notebook-13 status:", dataset_summary["status"])
print("Sequences:", len(sequences))
print("Unique recordings:", len(recordings))
print("Selected view:", SELECTED_VIEW)
print("Selected camera:", SELECTED_CAMERA_FILE)
print("Reported annotated duration, hours:", dataset_summary["total_duration_hours"])

display(recordings.head(10))
display(sequences.head(10))

dataset summary: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/dataset_summary.json -> exists=True
recording manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/recording_download_manifest_v1.csv -> exists=True
sequence/crop manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/procedurevrl_extraction_manifest_v1.csv -> exists=True
Notebook-13 status: completed
Sequences: 680
Unique recordings: 350
Selected view: v1
Selected camera: C10095_rgb.mp4
Reported annotated duration, hours: 40.20752777777778


,recording_name,selected_view,video_filename,video_remote_path,video_local_path,video_downloaded,num_sequence_crops
0,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724/C10095_rgb.mp4,False,2
1,nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253/C10095_rgb.mp4,False,2
2,nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736/C10095_rgb.mp4,False,2
3,nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620/C10095_rgb.mp4,False,2
4,nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239/C10095_rgb.mp4,False,2
5,nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-c13b_9011_user_id_2021-02-01_160915/C10095_rgb.mp4,False,2
6,nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9012-a16_9012_user_id_2021-02-01_162904/C10095_rgb.mp4,False,2
7,nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9012-a17_9012_user_id_2021-02-01_162209/C10095_rgb.mp4,False,2
8,nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9012-b06d_9012_user_id_2021-02-01_163713/C10095_rgb.mp4,False,2
9,nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9012-c06d_9012_user_id_2021-02-18_121034/C10095_rgb.mp4,False,2


,sequence_id,sequence_filename,split,activity,is_shared,toy_id,toy_name,recording_name,selected_view,video_filename,video_remote_path,video_local_path,video_downloaded,clip_start_frame_30fps,clip_end_frame_30fps_exclusive,clip_start_seconds,clip_end_seconds,clip_duration_seconds,ground_truth_path,ground_truth_length_30fps,num_background_frames,output_feature_name,feature_extraction_status
0,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,test,assembly,notshared,NaN,NaN,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724/C10095_rgb.mp4,False,4457,8070,148.566667,269.000000,120.433333,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,3613,0,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.npy,video_not_downloaded
1,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.txt,train,assembly,-,b06b,-,nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253/C10095_rgb.mp4,False,2833,6959,94.433333,231.966667,137.533333,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.txt,4126,0,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.npy,video_not_downloaded
2,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736.txt,train,assembly,-,b08c,-,nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736/C10095_rgb.mp4,False,4777,11525,159.233333,384.166667,224.933333,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736.txt,6748,0,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736.npy,video_not_downloaded
3,assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620,assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620.txt,test,assembly,notshared,NaN,NaN,nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620/C10095_rgb.mp4,False,4380,9978,146.000000,332.600000,186.600000,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620.txt,5598,0,assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620.npy,video_not_downloaded
4,assembly_nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239,assembly_nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239.txt,val,assembly,notshared,c03f,roller,nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011

## 8. Fetch exact remote sizes for the required 350 files

In [8]:
print(
    "Requesting repository file metadata. "
    "This can take a few minutes but downloads no video bytes."
)

repo_info_with_files = api.dataset_info(
    repo_id=REPO_ID,
    revision=PINNED_REVISION,
    files_metadata=True,
    token=True,
)

remote_metadata = {}

for sibling in repo_info_with_files.siblings:
    remote_path = sibling.rfilename

    if remote_path not in set(recordings["video_remote_path"]):
        continue

    size = sibling.size

    if size is None and sibling.lfs is not None:
        try:
            size = sibling.lfs.get("size")
        except AttributeError:
            size = getattr(sibling.lfs, "size", None)

    remote_metadata[remote_path] = {
        "remote_size_bytes": size,
        "remote_blob_id": sibling.blob_id,
    }

size_inventory = recordings.copy()
size_inventory["remote_size_bytes"] = size_inventory[
    "video_remote_path"
].map(
    lambda path: remote_metadata.get(path, {}).get(
        "remote_size_bytes"
    )
)
size_inventory["remote_blob_id"] = size_inventory[
    "video_remote_path"
].map(
    lambda path: remote_metadata.get(path, {}).get(
        "remote_blob_id"
    )
)

missing_size = size_inventory["remote_size_bytes"].isna()

print(
    "Required paths with remote size metadata:",
    int((~missing_size).sum()),
    "/",
    len(size_inventory),
)

if missing_size.any():
    display(
        size_inventory.loc[
            missing_size,
            ["recording_name", "video_remote_path"],
        ].head(100)
    )
    raise RuntimeError(
        "Remote size metadata is missing for required files."
    )

size_inventory["remote_size_bytes"] = (
    size_inventory["remote_size_bytes"].astype("int64")
)
size_inventory["remote_size_gib"] = (
    size_inventory["remote_size_bytes"] / 1024**3
)

size_inventory.to_csv(
    REMOTE_SIZE_MANIFEST_PATH,
    index=False,
)

total_remote_bytes = int(
    size_inventory["remote_size_bytes"].sum()
)

print(
    "Exact total size of all required v1 recordings:",
    f"{total_remote_bytes / 1024**3:.3f} GiB",
)
print(
    "Mean file size:",
    f"{size_inventory['remote_size_gib'].mean():.3f} GiB",
)
print(
    "Minimum file size:",
    f"{size_inventory['remote_size_gib'].min():.3f} GiB",
)
print(
    "Maximum file size:",
    f"{size_inventory['remote_size_gib'].max():.3f} GiB",
)
print("Saved:", REMOTE_SIZE_MANIFEST_PATH)

display(
    size_inventory[
        [
            "recording_name",
            "num_sequence_crops",
            "remote_size_gib",
            "video_remote_path",
        ]
    ]
    .sort_values("remote_size_gib")
    .head(20)
)

Requesting repository file metadata. This can take a few minutes but downloads no video bytes.
Required paths with remote size metadata: 350 / 350
Exact total size of all required v1 recordings: 384.522 GiB
Mean file size: 1.099 GiB
Minimum file size: 0.175 GiB
Maximum file size: 5.836 GiB
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/remote_size_inventory_v1.csv


,recording_name,num_sequence_crops,remote_size_gib,video_remote_path
334,nusar-2021_action_both_9084-b04c_9084_user_id_2021-02-25_143221,1,0.174741,recordings/nusar-2021_action_both_9084-b04c_9084_user_id_2021-02-25_143221/C10095_rgb.mp4
61,nusar-2021_action_both_9023-b05d_9023_user_id_2021-02-23_135325,1,0.223229,recordings/nusar-2021_action_both_9023-b05d_9023_user_id_2021-02-23_135325/C10095_rgb.mp4
63,nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,2,0.256879,recordings/nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136/C10095_rgb.mp4
202,nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,2,0.263341,recordings/nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244/C10095_rgb.mp4
342,nusar-2021_action_both_9085-c03d_9085_user_id_2021-02-22_174243,2,0.282362,recordings/nusar-2021_action_both_9085-c03d_9085_user_id_2021-02-22_174243/C10095_rgb.mp4
50,nusar-2021_action_both_9022-b03b_9022_user_id_2021-02-23_105258,2,0.292746,recordings/nusar-2021_action_both_9022-b03b_9022_user_id_2021-02-23_105258/C10095_rgb.mp4
224,nusar-2021_action_both_9055-b08b_9055_user_id_2021-02-24_104106,2,0.303863,recordings/nusar-2021_action_both_9055-b08b_9055_user_id_2021-02-24_104106/C10095_rgb.mp4
134,nusar-2021_action_both_9034-c13f_9034_user_id_2021-02-23_180813,2,0.344159,recordings/nusar-2021_action_both_9034-c13f_9034_user_id_2021-02-23_180813/C10095_rgb.mp4
257,nusar-2021_action_both_9064-a20_9064_user_id_2021-02-22_161628,2,0.344929,recordings/nusar-2021_action_both_9064-a20_9064_user_id_2021-02-22_161628/C10095_rgb.mp4
131,nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828,2,0.345736,recordings/nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828/C10095_rgb.mp4


## 9. Inspect existing local files and remaining storage requirement

In [9]:
def inspect_local_file(row):
    path = Path(row.video_local_path)
    exists = path.exists()
    local_size = path.stat().st_size if exists else 0
    expected_size = int(row.remote_size_bytes)

    return pd.Series({
        "local_exists": exists,
        "local_size_bytes": local_size,
        "size_matches_remote": (
            exists and local_size == expected_size
        ),
        "remaining_bytes_for_file": (
            0
            if exists and local_size == expected_size
            else expected_size
        ),
    })


local_state = size_inventory.apply(
    inspect_local_file,
    axis=1,
)

download_status = pd.concat(
    [
        size_inventory.reset_index(drop=True),
        local_state.reset_index(drop=True),
    ],
    axis=1,
)

download_status["status"] = np.select(
    [
        download_status["size_matches_remote"],
        download_status["local_exists"],
    ],
    [
        "complete",
        "local_size_mismatch",
    ],
    default="missing",
)

download_status["last_error"] = ""
download_status["last_checked_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

download_status.to_csv(
    DOWNLOAD_STATUS_PATH,
    index=False,
)

completed_count = int(
    download_status["size_matches_remote"].sum()
)
remaining_bytes = int(
    download_status["remaining_bytes_for_file"].sum()
)

disk = shutil.disk_usage(ASSEMBLY_ROOT)

print("Complete local recordings:", completed_count, "/", len(download_status))
print("Remaining full download:", f"{remaining_bytes / 1024**3:.3f} GiB")
print("Filesystem free space:", f"{disk.free / 1024**3:.3f} GiB")
print("Filesystem total space:", f"{disk.total / 1024**3:.3f} GiB")
print("Saved:", DOWNLOAD_STATUS_PATH)

if remaining_bytes > disk.free:
    print(
        "\nWARNING: the remaining complete v1 dataset is larger "
        "than the currently reported free filesystem space."
    )

display(
    download_status[
        [
            "recording_name",
            "status",
            "remote_size_gib",
            "local_size_bytes",
            "video_local_path",
        ]
    ]
    .sort_values(
        ["status", "remote_size_gib"],
        ascending=[True, True],
    )
    .head(30)
)

Complete local recordings: 0 / 350
Remaining full download: 384.522 GiB
Filesystem free space: 179.405 GiB
Filesystem total space: 235.676 GiB
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/download_status_v1.csv



,recording_name,status,remote_size_gib,local_size_bytes,video_local_path
334,nusar-2021_action_both_9084-b04c_9084_user_id_2021-02-25_143221,missing,0.174741,0,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9084-b04c_9084_user_id_2021-02-25_143221/C10095_rgb.mp4
61,nusar-2021_action_both_9023-b05d_9023_user_id_2021-02-23_135325,missing,0.223229,0,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9023-b05d_9023_user_id_2021-02-23_135325/C10095_rgb.mp4
63,nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,missing,0.256879,0,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136/C10095_rgb.mp4
202,nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,missing,0.263341,0,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244/C10095_rgb.mp4
342,nusar-2021_action_both_9085-c03d_9085_user_id_2021-02-22_174243,missing,0.282362,0,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9085-c03d_9085_user_id_2021-02-22_174243/C10095_rgb.mp4
50,nusar-2021_action_both_9022-b03b_9022_user_id_2021-02-23_105258,missing,0.292746,0,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9022-b03b_9022_user_id_2021-02-23_105258/C10095_rgb.mp4
224,nusar-2021_action_both_9055-b08b_9055_user_id_2021-02-24_104106,missing,0.303863,0,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9055-b08b_9055_user_id_2021-02-24_104106/C10095_rgb.mp4
134,nusar-2021_action_both_9034-c13f_9034_user_id_2021-02-23_180813,missing,0.344159,0,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9034-c13f_9034_user_id_2021-02-23_180813/C10095_rgb.mp4
257,nusar-2021_action_both_9064-a20_9064_user_id_2021-02-22_161628,missing,0.344929,0,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9064-a20_9064_user_id_2021-02-22_161628/C10095_rgb.mp4
131,nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828,missing,0.345736,0,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828/C10095_rgb.mp4


## 10. Select this run's download batch

In [10]:
incomplete = download_status[
    ~download_status["size_matches_remote"]
].copy()

complete = download_status[
    download_status["size_matches_remote"]
].copy()

if DOWNLOAD_MODE == "metadata_only":
    selected_batch = download_status.iloc[0:0].copy()

elif DOWNLOAD_MODE == "smoke":
    # Reuse already complete videos first, if present.
    existing_candidates = complete.sort_values(
        ["num_sequence_crops", "remote_size_bytes"],
        ascending=[False, True],
    )

    # Prefer small files with both assembly/disassembly sequence crops.
    missing_candidates = incomplete.copy()
    missing_candidates["has_two_crops"] = (
        missing_candidates["num_sequence_crops"] >= 2
    )
    missing_candidates = missing_candidates.sort_values(
        ["has_two_crops", "remote_size_bytes", "recording_name"],
        ascending=[False, True, True],
    )

    selected_batch = pd.concat(
        [existing_candidates, missing_candidates],
        ignore_index=True,
    ).drop_duplicates(
        subset=["recording_name"]
    ).head(SMOKE_RECORDINGS)

elif DOWNLOAD_MODE == "full":
    if not CONFIRM_FULL_DOWNLOAD:
        raise RuntimeError(
            "Full mode is selected, but "
            "CONFIRM_FULL_DOWNLOAD is False."
        )

    incomplete = incomplete.sort_values(
        "recording_name"
    )

    if FULL_BATCH_SIZE is None:
        selected_batch = incomplete
    else:
        selected_batch = incomplete.head(
            int(FULL_BATCH_SIZE)
        )

selected_batch = selected_batch.copy()
selected_batch["selected_download_mode"] = DOWNLOAD_MODE
selected_batch["selected_at_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

selected_batch.to_csv(
    SELECTED_BATCH_PATH,
    index=False,
)

batch_download_bytes = int(
    selected_batch.loc[
        ~selected_batch["size_matches_remote"],
        "remote_size_bytes",
    ].sum()
)

print("Mode:", DOWNLOAD_MODE)
print("Selected recordings:", len(selected_batch))
print(
    "New bytes required for this batch:",
    f"{batch_download_bytes / 1024**3:.3f} GiB",
)
print("Saved:", SELECTED_BATCH_PATH)

if batch_download_bytes > shutil.disk_usage(
    ASSEMBLY_ROOT
).free:
    raise RuntimeError(
        "The selected batch is larger than reported free space."
    )

display(
    selected_batch[
        [
            "recording_name",
            "num_sequence_crops",
            "status",
            "remote_size_gib",
            "video_remote_path",
        ]
    ]
)

Mode: smoke
Selected recordings: 2
New bytes required for this batch: 0.520 GiB
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/selected_download_batch_v1.csv


,recording_name,num_sequence_crops,status,remote_size_gib,video_remote_path
0,nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,2,missing,0.256879,recordings/nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136/C10095_rgb.mp4
1,nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,2,missing,0.263341,recordings/nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244/C10095_rgb.mp4


## 11. Download the selected files

Downloads are sequential. After every file, the persistent status CSV is updated.

Current `huggingface_hub` uses its local metadata/cache to avoid re-downloading complete files. If a local file exists with the wrong size, this notebook removes that incomplete target and requests the file again.

In [11]:
def save_status_frame(frame):
    frame = frame.copy()
    frame["last_checked_utc"] = (
        datetime.now(timezone.utc).isoformat()
    )
    frame.to_csv(DOWNLOAD_STATUS_PATH, index=False)


def update_one_status(
    frame,
    recording_name,
    *,
    status=None,
    error=None,
):
    mask = (
        frame["recording_name"] == recording_name
    )

    row = frame.loc[mask].iloc[0]
    local_path = Path(row["video_local_path"])
    exists = local_path.exists()
    local_size = (
        local_path.stat().st_size if exists else 0
    )
    expected_size = int(row["remote_size_bytes"])
    size_matches = (
        exists and local_size == expected_size
    )

    frame.loc[mask, "local_exists"] = exists
    frame.loc[mask, "local_size_bytes"] = local_size
    frame.loc[mask, "size_matches_remote"] = size_matches
    frame.loc[mask, "remaining_bytes_for_file"] = (
        0 if size_matches else expected_size
    )

    if status is None:
        if size_matches:
            status = "complete"
        elif exists:
            status = "local_size_mismatch"
        else:
            status = "missing"

    frame.loc[mask, "status"] = status

    if error is not None:
        frame.loc[mask, "last_error"] = error

    save_status_frame(frame)


if DOWNLOAD_MODE == "metadata_only":
    print("Metadata-only mode: no files downloaded.")

else:
    for row in tqdm(
        selected_batch.itertuples(index=False),
        total=len(selected_batch),
        desc="Assembly101 v1 files",
    ):
        recording_name = row.recording_name
        expected_size = int(row.remote_size_bytes)
        target_path = Path(row.video_local_path)

        if (
            target_path.exists()
            and target_path.stat().st_size == expected_size
        ):
            print(
                "\nAlready complete:",
                recording_name,
            )
            update_one_status(
                download_status,
                recording_name,
                status="complete",
                error="",
            )
            continue

        if target_path.exists():
            print(
                "\nRemoving incomplete/mismatched target:",
                target_path,
            )
            target_path.unlink()

        last_exception = None

        for attempt in range(
            1,
            MAX_DOWNLOAD_ATTEMPTS + 1,
        ):
            try:
                print(
                    f"\nDownloading {recording_name} "
                    f"(attempt {attempt}/{MAX_DOWNLOAD_ATTEMPTS})"
                )
                print(
                    "Expected size:",
                    f"{expected_size / 1024**3:.3f} GiB",
                )

                update_one_status(
                    download_status,
                    recording_name,
                    status="downloading",
                    error="",
                )

                returned_path = hf_hub_download(
                    repo_id=REPO_ID,
                    repo_type=REPO_TYPE,
                    revision=PINNED_REVISION,
                    filename=row.video_remote_path,
                    local_dir=ASSEMBLY_ROOT,
                    token=True,
                )

                returned_path = Path(returned_path)

                if returned_path.resolve() != target_path.resolve():
                    print(
                        "Returned path differs from manifest path:"
                    )
                    print(" returned:", returned_path)
                    print(" expected:", target_path)

                if not target_path.exists():
                    raise FileNotFoundError(
                        f"Downloaded target does not exist: {target_path}"
                    )

                actual_size = target_path.stat().st_size

                if actual_size != expected_size:
                    raise IOError(
                        f"Downloaded size mismatch for {recording_name}: "
                        f"{actual_size} != {expected_size}"
                    )

                update_one_status(
                    download_status,
                    recording_name,
                    status="complete",
                    error="",
                )

                print("Download verified:", target_path)
                last_exception = None
                break

            except Exception as exc:
                last_exception = exc
                error_text = (
                    f"{type(exc).__name__}: {exc}"
                )

                print("Download attempt failed:", error_text)

                update_one_status(
                    download_status,
                    recording_name,
                    status="failed",
                    error=error_text,
                )

                if attempt < MAX_DOWNLOAD_ATTEMPTS:
                    print(
                        "Sleeping before retry:",
                        RETRY_SLEEP_SECONDS,
                        "seconds",
                    )
                    time.sleep(RETRY_SLEEP_SECONDS)

        if last_exception is not None:
            print(
                "All attempts failed for:",
                recording_name,
            )

print("Persistent status:", DOWNLOAD_STATUS_PATH)

Assembly101 v1 files:   0%|          | 0/2 [00:00<?, ?it/s]


Expected size: 0.257 GiB


recordings/nusar-2021_action_both_9023-c(…): reconstructing file:   0%|          |  0.00B /  276MB            

recordings/nusar-2021_action_both_9023-c(…): downloading bytes:           |  0.00B            

Download verified: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136/C10095_rgb.mp4

Expected size: 0.263 GiB


recordings/nusar-2021_action_both_9052-c(…): reconstructing file:   0%|          |  0.00B /  283MB            

recordings/nusar-2021_action_both_9052-c(…): downloading bytes:           |  0.00B            

Download verified: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244/C10095_rgb.mp4
Persistent status: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/download_status_v1.csv


## 12. Refresh and verify the selected download batch

In [12]:
# Reload the persistent table in case this cell is run independently.
download_status = pd.read_csv(DOWNLOAD_STATUS_PATH)

for recording_name in selected_batch["recording_name"]:
    update_one_status(
        download_status,
        recording_name,
    )

selected_after = download_status[
    download_status["recording_name"].isin(
        set(selected_batch["recording_name"])
    )
].copy()

failed_selected = selected_after[
    ~selected_after["size_matches_remote"]
]

display(
    selected_after[
        [
            "recording_name",
            "status",
            "remote_size_gib",
            "local_size_bytes",
            "video_local_path",
            "last_error",
        ]
    ]
)

print(
    "Selected files complete:",
    int(selected_after["size_matches_remote"].sum()),
    "/",
    len(selected_after),
)

if len(selected_batch) > 0 and not failed_selected.empty:
    raise RuntimeError(
        "At least one selected video is incomplete. "
        "Rerun the download cell; completed files will be skipped."
    )

,recording_name,status,remote_size_gib,local_size_bytes,video_local_path,last_error
63,nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,complete,0.256879,275821971,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136/C10095_rgb.mp4,NaN
202,nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,complete,0.263341,282760168,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244/C10095_rgb.mp4,NaN


Selected files complete: 2 / 2


## 13. FFprobe all selected videos

In [13]:
def parse_fraction(value):
    if value in (None, "", "0/0"):
        return np.nan

    try:
        return float(Fraction(str(value)))
    except Exception:
        return np.nan


def ffprobe_video(path: Path):
    command = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0",
        "-show_entries",
        (
            "stream=codec_name,width,height,"
            "r_frame_rate,avg_frame_rate,"
            "nb_frames,duration:"
            "format=duration,size"
        ),
        "-of", "json",
        str(path),
    ]

    completed = subprocess.run(
        command,
        text=True,
        capture_output=True,
        check=False,
    )

    payload = (
        json.loads(completed.stdout)
        if completed.stdout.strip()
        else {}
    )

    streams = payload.get("streams", [])
    stream = streams[0] if streams else {}
    format_info = payload.get("format", {})

    duration_raw = (
        stream.get("duration")
        or format_info.get("duration")
    )
    size_raw = format_info.get("size")

    try:
        duration = float(duration_raw)
    except (TypeError, ValueError):
        duration = np.nan

    try:
        format_size = int(size_raw)
    except (TypeError, ValueError):
        format_size = np.nan

    return {
        "ffprobe_returncode": completed.returncode,
        "codec": stream.get("codec_name"),
        "width": stream.get("width"),
        "height": stream.get("height"),
        "r_frame_rate_raw": stream.get("r_frame_rate"),
        "avg_frame_rate_raw": stream.get("avg_frame_rate"),
        "r_frame_rate": parse_fraction(
            stream.get("r_frame_rate")
        ),
        "avg_frame_rate": parse_fraction(
            stream.get("avg_frame_rate")
        ),
        "nb_frames": stream.get("nb_frames"),
        "video_duration_seconds": duration,
        "ffprobe_format_size_bytes": format_size,
        "ffprobe_stderr": completed.stderr[-2000:],
    }


ffprobe_rows = []

for row in tqdm(
    selected_after.itertuples(index=False),
    total=len(selected_after),
    desc="FFprobe",
):
    path = Path(row.video_local_path)
    probe = ffprobe_video(path)

    ffprobe_rows.append({
        "recording_name": row.recording_name,
        "video_local_path": str(path),
        "remote_size_bytes": int(row.remote_size_bytes),
        "local_size_bytes": int(path.stat().st_size),
        **probe,
    })

current_ffprobe = pd.DataFrame(ffprobe_rows)

if FFPROBE_PATH.exists():
    old_ffprobe = pd.read_csv(FFPROBE_PATH)

    ffprobe_all = pd.concat(
        [old_ffprobe, current_ffprobe],
        ignore_index=True,
    ).drop_duplicates(
        subset=["recording_name"],
        keep="last",
    )
else:
    ffprobe_all = current_ffprobe

ffprobe_all.to_csv(
    FFPROBE_PATH,
    index=False,
)

display(current_ffprobe)

if not current_ffprobe.empty:
    if not (
        current_ffprobe["ffprobe_returncode"] == 0
    ).all():
        raise RuntimeError(
            "At least one selected video failed FFprobe."
        )

    if current_ffprobe[
        "video_duration_seconds"
    ].isna().any():
        raise RuntimeError(
            "At least one selected video has no readable duration."
        )

print("Saved:", FFPROBE_PATH)

FFprobe:   0%|          | 0/2 [00:00<?, ?it/s]

,recording_name,video_local_path,remote_size_bytes,local_size_bytes,ffprobe_returncode,codec,width,height,r_frame_rate_raw,avg_frame_rate_raw,r_frame_rate,avg_frame_rate,nb_frames,video_duration_seconds,ffprobe_format_size_bytes,ffprobe_stderr
0,nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136/C10095_rgb.mp4,275821971,275821971,0,h264,1920,1080,60/1,60/1,60.0,60.0,13349,222.483333,275821971,
1,nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244/C10095_rgb.mp4,282760168,282760168,0,h264,1920,1080,60/1,60/1,60.0,60.0,12317,205.283333,282760168,


Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/ffprobe_v1.csv


## 14. Validate annotation crop intervals against video durations

Notebook 13 converted annotation frames to seconds using 30 fps. The selected raw video is nominally 60 fps, so seconds—not raw frame numbers—are used for alignment.

In [14]:
selected_recording_names = set(
    selected_after["recording_name"]
)

selected_sequences = sequences[
    sequences["recording_name"].isin(
        selected_recording_names
    )
].copy()

duration_table = current_ffprobe[
    [
        "recording_name",
        "video_duration_seconds",
        "r_frame_rate",
        "avg_frame_rate",
        "width",
        "height",
        "codec",
    ]
].copy()

alignment = selected_sequences.merge(
    duration_table,
    on="recording_name",
    how="left",
    validate="many_to_one",
)

alignment["clip_start_valid"] = (
    alignment["clip_start_seconds"] >= 0
)
alignment["clip_end_after_start"] = (
    alignment["clip_end_seconds"]
    > alignment["clip_start_seconds"]
)
alignment["clip_end_within_video"] = (
    alignment["clip_end_seconds"]
    <= (
        alignment["video_duration_seconds"]
        + DURATION_TOLERANCE_SECONDS
    )
)
alignment["seconds_past_video_end"] = np.maximum(
    alignment["clip_end_seconds"]
    - alignment["video_duration_seconds"],
    0.0,
)
alignment["alignment_valid"] = (
    alignment["clip_start_valid"]
    & alignment["clip_end_after_start"]
    & alignment["clip_end_within_video"]
)

alignment.to_csv(
    ALIGNMENT_PATH,
    index=False,
)

print("Selected sequence crops:", len(alignment))
print(
    "Valid crop intervals:",
    int(alignment["alignment_valid"].sum()),
    "/",
    len(alignment),
)
print(
    "Maximum seconds past video end:",
    (
        float(
            alignment[
                "seconds_past_video_end"
            ].max()
        )
        if len(alignment)
        else 0.0
    ),
)

display(
    alignment[
        [
            "sequence_id",
            "split",
            "activity",
            "recording_name",
            "clip_start_seconds",
            "clip_end_seconds",
            "clip_duration_seconds",
            "video_duration_seconds",
            "r_frame_rate",
            "alignment_valid",
            "seconds_past_video_end",
        ]
    ]
)

if len(alignment) > 0 and not alignment[
    "alignment_valid"
].all():
    display(
        alignment.loc[
            ~alignment["alignment_valid"]
        ]
    )
    raise RuntimeError(
        "At least one annotation crop falls outside its video."
    )

print("Saved:", ALIGNMENT_PATH)

Selected sequence crops: 4
Valid crop intervals: 4 / 4
Maximum seconds past video end: 0.0


,sequence_id,split,activity,recording_name,clip_start_seconds,clip_end_seconds,clip_duration_seconds,video_duration_seconds,r_frame_rate,alignment_valid,seconds_past_video_end
0,assembly_nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,test,assembly,nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,87.666667,212.966667,125.300000,222.483333,60.0,True,0.0
1,assembly_nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,train,assembly,nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,97.400000,203.666667,106.266667,205.283333,60.0,True,0.0
2,disassembly_nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,test,disassembly,nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,5.166667,84.400000,79.233333,222.483333,60.0,True,0.0
3,disassembly_nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,train,disassembly,nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,9.933333,92.466667,82.533333,205.283333,60.0,True,0.0


Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/crop_alignment_validation_v1.csv


## 15. Create short smoke preview crops

In [15]:
def safe_output_name(value):
    return re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(value),
    )


crop_rows = []

if (
    DOWNLOAD_MODE != "smoke"
    or not CREATE_SMOKE_CROPS
):
    print(
        "Smoke-crop creation is disabled for this run."
    )

else:
    for row in tqdm(
        alignment.itertuples(index=False),
        total=len(alignment),
        desc="Smoke crops",
    ):
        input_path = Path(row.video_local_path)
        output_path = (
            SMOKE_CROP_ROOT
            / f"{safe_output_name(row.sequence_id)}__preview.mp4"
        )

        preview_duration = min(
            float(SMOKE_CROP_MAX_SECONDS),
            float(row.clip_duration_seconds),
            max(
                0.0,
                float(row.video_duration_seconds)
                - float(row.clip_start_seconds),
            ),
        )

        if preview_duration <= 0:
            crop_rows.append({
                "sequence_id": row.sequence_id,
                "recording_name": row.recording_name,
                "activity": row.activity,
                "input_video": str(input_path),
                "output_crop": str(output_path),
                "crop_start_seconds": float(
                    row.clip_start_seconds
                ),
                "requested_duration_seconds": preview_duration,
                "status": "invalid_duration",
                "ffmpeg_returncode": -1,
                "error": "Preview duration is non-positive.",
            })
            continue

        command = [
            "ffmpeg",
            "-hide_banner",
            "-loglevel", "error",
            "-y",
            "-ss", f"{float(row.clip_start_seconds):.6f}",
            "-i", str(input_path),
            "-t", f"{preview_duration:.6f}",
            "-map", "0:v:0",
            "-an",
            "-vf",
            (
                f"scale=-2:{int(SMOKE_CROP_HEIGHT)},"
                "fps=30"
            ),
            "-c:v", "libx264",
            "-preset", "veryfast",
            "-crf", str(int(SMOKE_CROP_CRF)),
            "-movflags", "+faststart",
            str(output_path),
        ]

        completed = subprocess.run(
            command,
            text=True,
            capture_output=True,
            check=False,
        )

        status = (
            "created"
            if completed.returncode == 0
            and output_path.exists()
            and output_path.stat().st_size > 0
            else "failed"
        )

        crop_rows.append({
            "sequence_id": row.sequence_id,
            "recording_name": row.recording_name,
            "activity": row.activity,
            "input_video": str(input_path),
            "output_crop": str(output_path),
            "crop_start_seconds": float(
                row.clip_start_seconds
            ),
            "requested_duration_seconds": preview_duration,
            "status": status,
            "ffmpeg_returncode": completed.returncode,
            "output_size_mb": (
                output_path.stat().st_size / 1024**2
                if output_path.exists()
                else np.nan
            ),
            "error": completed.stderr[-3000:],
        })

smoke_crop_manifest = pd.DataFrame(crop_rows)
smoke_crop_manifest.to_csv(
    SMOKE_CROP_MANIFEST_PATH,
    index=False,
)

display(smoke_crop_manifest)

if (
    DOWNLOAD_MODE == "smoke"
    and CREATE_SMOKE_CROPS
    and len(alignment) > 0
):
    if smoke_crop_manifest.empty:
        raise RuntimeError(
            "No smoke crops were attempted."
        )

    if not (
        smoke_crop_manifest["status"] == "created"
    ).all():
        raise RuntimeError(
            "At least one smoke crop failed."
        )

print("Saved:", SMOKE_CROP_MANIFEST_PATH)

Smoke crops:   0%|          | 0/4 [00:00<?, ?it/s]

,sequence_id,recording_name,activity,input_video,output_crop,crop_start_seconds,requested_duration_seconds,status,ffmpeg_returncode,output_size_mb,error
0,assembly_nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,assembly,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/smoke_crops/assembly_nusar-2021_action_both_9023-c04b_9023_user_id_2021-02...,87.666667,12.0,created,0,0.116900,
1,assembly_nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,assembly,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/smoke_crops/assembly_nusar-2021_action_both_9052-c03d_9052_user_id_2021-02...,97.400000,12.0,created,0,0.098106,
2,disassembly_nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,disassembly,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/smoke_crops/disassembly_nusar-2021_action_both_9023-c04b_9023_user_id_2021...,5.166667,12.0,created,0,0.098470,
3,disassembly_nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,disassembly,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/smoke_crops/disassembly_nusar-2021_action_both_9052-c03d_9052_user_id_2021...,9.933333,12.0,created,0,0.087941,


Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/smoke_crop_manifest_v1.csv


## 16. FFprobe and optionally display the first smoke crop

In [16]:
smoke_crop_probe_rows = []

if (
    "smoke_crop_manifest" in globals()
    and not smoke_crop_manifest.empty
):
    for row in smoke_crop_manifest.itertuples(
        index=False
    ):
        if row.status != "created":
            continue

        path = Path(row.output_crop)
        probe = ffprobe_video(path)

        smoke_crop_probe_rows.append({
            "sequence_id": row.sequence_id,
            "output_crop": str(path),
            **probe,
        })

smoke_crop_probe = pd.DataFrame(
    smoke_crop_probe_rows
)
display(smoke_crop_probe)

if not smoke_crop_probe.empty:
    assert (
        smoke_crop_probe["ffprobe_returncode"] == 0
    ).all()

    from IPython.display import Video, display as ipy_display

    first_crop = Path(
        smoke_crop_probe.iloc[0]["output_crop"]
    )

    print("Displaying first crop:", first_crop)
    ipy_display(
        Video(
            str(first_crop),
            embed=False,
            width=640,
        )
    )
else:
    print("No smoke crop to display in this mode.")

,sequence_id,output_crop,ffprobe_returncode,codec,width,height,r_frame_rate_raw,avg_frame_rate_raw,r_frame_rate,avg_frame_rate,nb_frames,video_duration_seconds,ffprobe_format_size_bytes,ffprobe_stderr
0,assembly_nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/smoke_crops/assembly_nusar-2021_action_both_9023-c04b_9023_user_id_2021-02...,0,h264,640,360,30/1,30/1,30.0,30.0,360,12.0,122579,
1,assembly_nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/smoke_crops/assembly_nusar-2021_action_both_9052-c03d_9052_user_id_2021-02...,0,h264,640,360,30/1,30/1,30.0,30.0,360,12.0,102872,
2,disassembly_nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/smoke_crops/disassembly_nusar-2021_action_both_9023-c04b_9023_user_id_2021...,0,h264,640,360,30/1,30/1,30.0,30.0,360,12.0,103253,
3,disassembly_nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/smoke_crops/disassembly_nusar-2021_action_both_9052-c03d_9052_user_id_2021...,0,h264,640,360,30/1,30/1,30.0,30.0,360,12.0,92213,


Displaying first crop: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/smoke_crops/assembly_nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136__preview.mp4


## 17. Refresh global progress after this run

In [17]:
# Refresh every required recording from the filesystem.
download_status = pd.read_csv(
    DOWNLOAD_STATUS_PATH
)

for recording_name in download_status[
    "recording_name"
].tolist():
    update_one_status(
        download_status,
        recording_name,
    )

download_status = pd.read_csv(
    DOWNLOAD_STATUS_PATH
)

complete_total = int(
    download_status["size_matches_remote"].sum()
)
complete_bytes = int(
    download_status.loc[
        download_status["size_matches_remote"],
        "remote_size_bytes",
    ].sum()
)
remaining_bytes = int(
    download_status.loc[
        ~download_status["size_matches_remote"],
        "remote_size_bytes",
    ].sum()
)

print("Complete recordings:", complete_total, "/", len(download_status))
print("Complete bytes:", f"{complete_bytes / 1024**3:.3f} GiB")
print("Remaining bytes:", f"{remaining_bytes / 1024**3:.3f} GiB")
print("Progress:", f"{100.0 * complete_total / len(download_status):.2f}%")

display(
    download_status["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="num_recordings")
)

Complete recordings: 2 / 350
Complete bytes: 0.520 GiB
Remaining bytes: 384.002 GiB
Progress: 0.57%


,status,num_recordings
0,missing,348
1,complete,2


## 18. Save the run summary

In [18]:
selected_complete_count = int(
    selected_after["size_matches_remote"].sum()
)

alignment_valid_count = (
    int(alignment["alignment_valid"].sum())
    if "alignment" in globals()
    else 0
)

smoke_crop_count = (
    int(
        (
            smoke_crop_manifest["status"] == "created"
        ).sum()
    )
    if (
        "smoke_crop_manifest" in globals()
        and not smoke_crop_manifest.empty
    )
    else 0
)

summary = {
    "status": "completed",
    "dataset": "Assembly101",
    "view": SELECTED_VIEW,
    "camera_file": SELECTED_CAMERA_FILE,
    "repo_id": REPO_ID,
    "pinned_revision": PINNED_REVISION,
    "download_mode": DOWNLOAD_MODE,
    "num_required_recordings": int(
        len(download_status)
    ),
    "total_required_size_gib": (
        total_remote_bytes / 1024**3
    ),
    "num_recordings_complete_after_run": (
        complete_total
    ),
    "complete_size_gib_after_run": (
        complete_bytes / 1024**3
    ),
    "remaining_size_gib_after_run": (
        remaining_bytes / 1024**3
    ),
    "selected_batch": {
        "num_recordings": int(
            len(selected_batch)
        ),
        "new_download_size_gib": (
            batch_download_bytes / 1024**3
        ),
        "num_complete": selected_complete_count,
    },
    "video_validation": {
        "num_ffprobed_in_this_run": int(
            len(current_ffprobe)
        ),
        "num_ffprobe_failures": (
            int(
                (
                    current_ffprobe[
                        "ffprobe_returncode"
                    ] != 0
                ).sum()
            )
            if not current_ffprobe.empty
            else 0
        ),
    },
    "crop_alignment": {
        "num_sequences_checked": int(
            len(alignment)
        ),
        "num_valid": alignment_valid_count,
        "num_invalid": (
            int(
                (
                    ~alignment["alignment_valid"]
                ).sum()
            )
            if len(alignment)
            else 0
        ),
        "duration_tolerance_seconds": (
            DURATION_TOLERANCE_SECONDS
        ),
    },
    "smoke_crops": {
        "enabled": bool(
            DOWNLOAD_MODE == "smoke"
            and CREATE_SMOKE_CROPS
        ),
        "num_created": smoke_crop_count,
        "max_preview_seconds": (
            SMOKE_CROP_MAX_SECONDS
        ),
    },
    "paths": {
        "remote_size_inventory": str(
            REMOTE_SIZE_MANIFEST_PATH
        ),
        "download_status": str(
            DOWNLOAD_STATUS_PATH
        ),
        "selected_batch": str(
            SELECTED_BATCH_PATH
        ),
        "ffprobe": str(FFPROBE_PATH),
        "crop_alignment": str(
            ALIGNMENT_PATH
        ),
        "smoke_crop_manifest": str(
            SMOKE_CROP_MANIFEST_PATH
        ),
        "smoke_crop_root": str(
            SMOKE_CROP_ROOT
        ),
    },
    "next_step": (
        "After a successful smoke run, continue the resumable full v1 "
        "download in controlled batches. Once the required recordings "
        "are available, extract temporally aligned ProcedureVRL and "
        "CLIP-like visual features for each of the 680 sequence crops."
    ),
}

SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2))
print("\nSaved:", SUMMARY_PATH)

if DOWNLOAD_MODE == "smoke":
    print("\nSmoke run completed.")
    print("Next full-download settings:")
    print('DOWNLOAD_MODE = "full"')
    print("CONFIRM_FULL_DOWNLOAD = True")
    print("FULL_BATCH_SIZE = 10")
    print(
        "Rerun full mode until "
        "num_recordings_complete_after_run == "
        "num_required_recordings."
    )

elif DOWNLOAD_MODE == "full":
    if complete_total < len(download_status):
        print(
            "\nFull download is not complete yet. "
            "Rerun the notebook to download the next batch."
        )
    else:
        print(
            "\nAll required v1 recordings are complete."
        )
        print("Next notebook:")
        print(
            "15_assembly101_procedurevrl_and_clip_feature_extraction_COLAB.ipynb"
        )

{
  "status": "completed",
  "dataset": "Assembly101",
  "view": "v1",
  "camera_file": "C10095_rgb.mp4",
  "repo_id": "cvml-nus/assembly101",
  "pinned_revision": "bfc15ea5e3f0bc8f8c232af6c1b45aa137a9d967",
  "download_mode": "smoke",
  "num_required_recordings": 350,
  "total_required_size_gib": 384.5220845518634,
  "num_recordings_complete_after_run": 2,
  "complete_size_gib_after_run": 0.5202201558277011,
  "remaining_size_gib_after_run": 384.00186439603567,
  "selected_batch": {
    "num_recordings": 2,
    "new_download_size_gib": 0.5202201558277011,
    "num_complete": 2
  },
  "video_validation": {
    "num_ffprobed_in_this_run": 2,
    "num_ffprobe_failures": 0
  },
  "crop_alignment": {
    "num_sequences_checked": 4,
    "num_valid": 4,
    "num_invalid": 0,
    "duration_tolerance_seconds": 1.0
  },
  "smoke_crops": {
    "enabled": true,
    "num_created": 4,
    "max_preview_seconds": 12.0
  },
  "paths": {
    "remote_size_inventory": "/content/drive/MyDrive/mmf_tas_lab_

## Expected smoke-run success

The final summary should show:

```text
status: completed
download_mode: smoke
num_required_recordings: 350
selected_batch.num_recordings: 2
selected_batch.num_complete: 2
video_validation.num_ffprobe_failures: 0
crop_alignment.num_invalid: 0
smoke_crops.num_created: > 0
```

The exact total required size is discovered from the pinned Hugging Face repository metadata and is not hard-coded.